# Submission P06: A with uncertainty ranks 9–14 flipped (both edges)

| | |
|---|---|
| Submitted file | `P06.csv` |
| Submitted (UTC) | 2026-10-07 18:15 |
| Public score | 0.93370 (169/181) |
| Rule | A with uncertainty ranks 9–14 flipped (both edges) |

One of the 22 tickets of the final-selection portfolio (E23, `solution/portfolio25.py`): the competition lets a team select up to 25 final submissions and the private leaderboard counts the best of them. Submitted as `P06.csv` from the Kaggle notebook [kacchi-aloo-portfolio](https://www.kaggle.com/code/hosen42/kacchi-aloo-portfolio) v1 (source: `solution/kaggle_portfolio_kernel/`), which writes all 22 tickets.

**Run it:** on Kaggle, upload this notebook and add the competition data (*Add Input → Competitions → The Great Kacchi Aloo Mystery*), then *Save & Run All* and submit `submission.csv`. Locally, run it from this folder (it reads `../data/train.csv` and `../data/test.csv`). The last cell checks the output against the file that was actually submitted.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [3]:
from sklearn.model_selection import StratifiedKFold


def sharp_band_probabilities(tr, te, bins=(0.0, 0.01, 0.035, 0.1, 0.4, np.inf), repeats=10, seed=0):
    """P(y=1) per test row = the sharp band's out-of-fold error rate at that distance from the nearest edge.
    Test rows inside the training gap around an edge get P from a uniform prior on where the cut lies."""
    r, y = tr.apg.values, tr.y.values
    lo, hi = midcut_cuts(r, y)
    d_oof, e_oof = [], []
    for k in range(repeats):
        for t, v in StratifiedKFold(5, shuffle=True, random_state=seed + k).split(tr, y):
            l2, h2 = midcut_cuts(r[t], y[t])
            d_oof.append(np.minimum(abs(r[v] - l2), abs(r[v] - h2)))
            e_oof.append(band(r[v], l2, h2) != y[v])
    d_oof, e_oof = np.concatenate(d_oof), np.concatenate(e_oof)
    idx = np.digitize(d_oof, bins[1:-1])
    err = np.array([(e_oof[idx == b].sum() + 0.5) / ((idx == b).sum() + 1.0) for b in range(len(bins) - 1)])
    below_lo, above_lo = r[r < lo].max(), r[(r >= lo) & (r < 1.5)].min()
    below_hi, above_hi = r[(r <= hi) & (r > 1.5)].max(), r[r > hi].min()
    x = te.apg.values
    d = np.minimum(abs(x - lo), abs(x - hi))
    e = err[np.digitize(d, bins[1:-1])]
    P = np.where(band(x, lo, hi) == 1, 1 - e, e)
    q_lo = np.clip((x - below_lo) / (above_lo - below_lo), 0, 1)
    q_hi = np.clip((above_hi - x) / (above_hi - below_hi), 0, 1)
    g_lo, g_hi = (x > below_lo) & (x < above_lo), (x > below_hi) & (x < above_hi)
    P = np.where(g_lo, q_lo * (1 - err[0]) + (1 - q_lo) * err[0], P)
    P = np.where(g_hi, q_hi * (1 - err[0]) + (1 - q_hi) * err[0], P)
    return P, (lo, hi), err


P, (LO, HI), err = sharp_band_probabilities(tr, te)
print(f"cut-points {LO:.5f} / {HI:.5f}; out-of-fold error rate by distance to the edge "
      f"(0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): {np.round(err, 3)}")

cut-points 0.97068 / 2.00983; out-of-fold error rate by distance to the edge (0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): [0.127 0.374 0.148 0.022 0.   ]


In [4]:
SIDE, START, WIDTH = 'both', 8, 6   # ticket P06: flip uncertainty ranks 9-14 (both edges)
A = band(te.apg.values, LO, HI)
order = np.lexsort((np.minimum(abs(te.apg.values - LO), abs(te.apg.values - HI)), np.abs(P - 0.5)))
low = te.apg.values < 1.5
o = {"both": order, "lower": order[low[order]], "upper": order[~low[order]]}[SIDE]
pred = A.copy()
pred[o[START:START + WIDTH]] ^= 1
print(te.assign(P=P.round(3), A=A, ticket=pred).loc[A != pred, ["wedding_id", "guests", "aloo_count", "apg", "P", "A", "ticket"]]
      .sort_values("apg").to_string(index=False))

wedding_id guests aloo_count      apg     P  A  ticket
     W1105    186        176 0.946237 0.374  0       1
     W0851   1109       1050 0.946799 0.374  0       1
     W1135   1467       2913 1.985685 0.626  1       0
     W0865   1319       2627 1.991660 0.626  1       0
     W0974    442        896 2.027149 0.374  0       1
     W1058    432        879 2.034722 0.374  0       1


In [5]:
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": np.asarray(pred).astype(int)})
sub.to_csv("submission.csv", index=False)
fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
print(f"wrote submission.csv: {sub.went_back_for_seconds.sum()} ones / {len(sub)} rows")
print(f"identical to the submitted file 'P06.csv': {fp == '7997d2a083e7385d9e1716a6a191a0b2'}")

wrote submission.csv: 184 ones / 400 rows
identical to the submitted file 'P06.csv': True
